# Interpretable Predictions for Steam Game Recommendations

**Author:** Michael Theophanopoulos  
**Purpose:** Binary classifier with counterfactual explanations  
**Last Updated:** 2025-10-29

---

## Notebook Structure

1. Data Loading & Exploration
2. Feature Engineering
3. Model Training
4. Prediction Function (with Counterfactuals)
5. Model Evaluation

In [ ]:
# ============================================================================
# IMPORTS & CONFIGURATION
# ============================================================================

# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import sys
from datetime import datetime

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ============================================================================
# LOGGING CONFIGURATION (Enterprise-level)
# ============================================================================

def setup_logger(name: str = 'steam_predictions', level: int = logging.INFO) -> logging.Logger:
    """
    Configure enterprise-level logging for the notebook.
    
    Best practices implemented:
    - Structured log format with timestamp, level, and message
    - Both console and file output
    - Proper log rotation prevention (overwrites for notebooks)
    - Clear, hierarchical logger naming
    
    Parameters
    ----------
    name : str
        Logger name (hierarchical, e.g., 'steam_predictions.features')
    level : int
        Logging level (DEBUG, INFO, WARNING, ERROR, CRITICAL)
        
    Returns
    -------
    logging.Logger
        Configured logger instance
    """
    # Create logger
    logger = logging.getLogger(name)
    logger.setLevel(level)
    
    # Prevent duplicate handlers in notebook environment
    if logger.handlers:
        logger.handlers.clear()
    
    # Create formatters
    console_formatter = logging.Formatter(
        fmt='%(levelname)-8s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    file_formatter = logging.Formatter(
        fmt='%(asctime)s | %(name)-25s | %(levelname)-8s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Console handler (for notebook output)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(level)
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)
    
    # File handler (for audit trail)
    log_file = Path('logs') / f'steam_predictions_{datetime.now():%Y%m%d}.log'
    log_file.parent.mkdir(exist_ok=True)
    
    file_handler = logging.FileHandler(log_file, mode='a', encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)  # Capture everything in file
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)
    
    # Prevent propagation to root logger
    logger.propagate = False
    
    return logger

# Initialize main logger
logger = setup_logger('steam_predictions', level=logging.INFO)

logger.info("="*60)
logger.info("Steam Game Recommendation Prediction System")
logger.info(f"Notebook initialized at {datetime.now():%Y-%m-%d %H:%M:%S}")
logger.info("="*60)

---

## 0. Data Scraping

Scape the data from https://store.steampowered.com/appreviews/ and store them in a csv file.

In [48]:
import requests
import pandas as pd
import time
from pathlib import Path
from typing import List, Dict, Optional
import logging

In [49]:
def scrape_steam_reviews(
    app_id: int,
    max_reviews: int = 50000,
    language: str = 'english',
    reviews_per_page: int = 100
) -> List[Dict]:
    """
    Scrape game reviews from Steam API.
    
    Parameters
    ----------
    app_id : int
        Steam application ID (default: 1245620 for Elden Ring)
    max_reviews : int
        Maximum number of reviews to collect
    language : str
        Review language filter
    reviews_per_page : int
        Number of reviews per API request (max 100)
        
    Returns
    -------
    List[Dict]
        List of review dictionaries
    """
    all_reviews = []
    cursor = '*'
    start_time = time.time()
    
    logging.info(f"Starting fetching reviews for app_id={app_id}...")
    
    while len(all_reviews) < max_reviews:
        url = f'https://store.steampowered.com/appreviews/{app_id}'
        params = {
            'json': 1,
            'language': language,
            'cursor': cursor,
            'num_per_page': reviews_per_page,
            'filter': 'recent'
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if not data.get('reviews'):
                logging.warning("No more reviews available")
                break
            
            all_reviews.extend(data['reviews'])
            cursor = data.get('cursor')
            
            if not cursor:
                logging.warning("No cursor returned, ending pagination")
                break
            
            # Rate limiting: 0.3s between requests
            time.sleep(0.3)
            
        except requests.exceptions.RequestException as e:
            logging.error(f"Request failed: {e}")
            time.sleep(5)
    
    final_reviews = all_reviews[:max_reviews]
    elapsed_time = time.time() - start_time
    
    logging.info(f"Fetched {len(final_reviews)} reviews in {elapsed_time:.2f} seconds")
    
    return final_reviews

In [50]:
def process_reviews(reviews: List[Dict], game_data: Dict = None) -> pd.DataFrame:
    """
    Convert raw review data to structured DataFrame.
    
    Parameters
    ----------
    reviews : List[Dict]
        Raw review data from Steam API
    game_data : Dict, optional
        Game metadata from appdetails API (price, genre, description, etc.)
        
    Returns
    -------
    pd.DataFrame
        Structured dataset with selected features for binary classification
    """
    data = []
    
    for review in reviews:
        author = review.get('author', {})
        
        record = {
            # Target variable
            'recommended': review.get('voted_up', False),
            
            # User attributes (descriptive features)
            'user_id': author.get('steamid', ''),
            'user_games_owned': author.get('num_games_owned', 0),
            'user_num_reviews': author.get('num_reviews', 0),
            
            # User-Game interaction (numeric features)
            'playtime_at_review_hours': author.get('playtime_at_review', 0) / 60,
            'playtime_total_hours': author.get('playtime_forever', 0) / 60,
            'playtime_recent_hours': author.get('playtime_last_two_weeks', 0) / 60,
            
            # Review metadata (categorical features)
            'received_free': review.get('received_for_free', False),
            'steam_purchase': review.get('steam_purchase', True),
            'written_early_access': review.get('written_during_early_access', False),
            
            # Review engagement (numeric features)
            'votes_helpful': review.get('votes_up', 0),
            'votes_funny': review.get('votes_funny', 0),
            'weighted_vote_score': review.get('weighted_vote_score', 0.0),
            'comment_count': review.get('comment_count', 0),
            
            # Temporal features
            'timestamp_created': review.get('timestamp_created', 0),
            'timestamp_updated': review.get('timestamp_updated', 0),
            
            # Text features (free text)
            'review_text': review.get('review', ''),
            'review_length': len(review.get('review', '')),
            
            # Unique identifiers
            'review_id': review.get('recommendationid', '')
        }
        
        # Add game features if game_data provided
        if game_data:
            record.update({
                'game_id': game_data.get('steam_appid', ''),
                'game_name': game_data.get('name', ''),
                'game_price': game_data.get('price_overview', {}).get('final', 0) / 100 if game_data.get('price_overview') else 0,
                'game_is_free': game_data.get('is_free', False),
                'game_genre': ','.join([g['description'] for g in game_data.get('genres', [])]),
                'game_categories': ','.join([c['description'] for c in game_data.get('categories', [])]),
                'game_developer': ','.join(game_data.get('developers', [])),
                'game_publisher': ','.join(game_data.get('publishers', [])),
                'game_description': game_data.get('short_description', ''),
                'game_required_age': game_data.get('required_age', 0),
            })
        
        data.append(record)
    
    df = pd.DataFrame(data)
    
    # Feature engineering
    df['playtime_ratio'] = df['playtime_at_review_hours'] / (df['playtime_total_hours'] + 1)  # Avoid division by zero
    df['review_engagement_score'] = df['votes_helpful'] + df['votes_funny'] * 0.5  # Weighted engagement
    df['experienced_gamer'] = df['user_games_owned'] > 50  # Binary feature
    df['active_reviewer'] = df['user_num_reviews'] > 5  # Binary feature
    
    logging.info(f"Processed {len(df)} reviews into DataFrame with {len(df.columns)} features")
    
    return df

In [51]:
# Configuration
APP_ID = 578080  # PUBG
MAX_REVIEWS = 50000
OUTPUT_FILE = 'steam_reviews.csv'

# Collect reviews
reviews = scrape_steam_reviews(app_id=APP_ID, max_reviews=MAX_REVIEWS)

INFO:root:Starting fetching reviews for app_id=578080...
INFO:root:Fetched 50000 reviews in 384.34 seconds


In [52]:
# Process and save
df = process_reviews(reviews)
df.to_csv(OUTPUT_FILE, index=False)
logging.info(f"Dataset saved to {OUTPUT_FILE}")

INFO:root:Processed 50000 reviews into DataFrame with 23 features
INFO:root:Dataset saved to steam_reviews.csv


In [53]:
# Data summary
print(f"Dataset shape: {df.shape}")
print(f"Positive reviews: {df['recommended'].sum()} ({df['recommended'].mean()*100:.1f}%)")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (50000, 23)
Positive reviews: 34833 (69.7%)

First few rows:


,recommended,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,received_free,steam_purchase,written_early_access,...,comment_count,timestamp_created,timestamp_updated,review_text,review_length,review_id,playtime_ratio,review_engagement_score,experienced_gamer,active_reviewer
0,True,76561197990088609,0,11,80.316667,80.316667,0.000000,False,True,False,...,0,1761705575,1761705575,"The game play is solid, I started when it was ...",370,207842607,0.987702,0.0,False,True
1,True,76561198347370385,78,2,351.566667,352.650000,25.266667,False,True,False,...,0,1761700391,1761700391,funny,5,207838132,0.994109,0.0,True,False
2,False,76561198372979773,280,26,13.900000,13.900000,1.583333,False,True,False,...,0,1761686608,1761686608,shi8,4,207823715,0.932886,4.5,True,True
3,False,76561198027390661,189,13,168.216667,168.216667,0.000000,False,True,False,...,0,1761685221,1761685221,Litteraly WHO plays this game anymore,37,207822055,0.994090,0.0,True,True
4,True,76561198310415102,61,6,218.083333,218.716667,7.516667,False,True,False,...,0,1761670033,1761670033,pog,3,207803041,0.992566,0.0,True,True


---

## 1. Data Loading & Exploration

Load the preprocessed Steam reviews dataset and perform exploratory analysis.

In [ ]:
# ============================================================================
# DATA LOADING
# ============================================================================

# File paths
DATA_FILE = Path('steam_reviews.csv')

# Validate file exists
if not DATA_FILE.exists():
    logger.error(f"Data file not found: {DATA_FILE}")
    raise FileNotFoundError(
        f"Data file not found: {DATA_FILE}. "
        "Please run data collection first."
    )

# Load data
logger.info(f"Loading data from {DATA_FILE}")
df = pd.read_csv(DATA_FILE)
logger.info(f"Loaded {len(df):,} reviews with {df.shape[1]} columns")

# Display shape
print(f"\nDataset shape: {df.shape}")
df.head()

In [55]:
# Dataset overview
print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Total records: {len(df):,}")
print(f"Features: {df.shape[1]}")
print(f"\nTarget distribution:")
print(f"  Positive reviews: {df['recommended'].sum():,} ({df['recommended'].mean()*100:.2f}%)")
print(f"  Negative reviews: {(~df['recommended']).sum():,} ({(~df['recommended']).mean()*100:.2f}%)")
print("\n" + "="*60)

DATASET SUMMARY
Total records: 50,000
Features: 23

Target distribution:
  Positive reviews: 34,833 (69.67%)
  Negative reviews: 15,167 (30.33%)



In [56]:
# Data quality check
print("DATA QUALITY REPORT")
print("="*60)
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe()

DATA QUALITY REPORT

Missing values:
recommended                   0
user_id                       0
user_games_owned              0
user_num_reviews              0
playtime_at_review_hours      0
playtime_total_hours          0
playtime_recent_hours         0
received_free                 0
steam_purchase                0
written_early_access          0
votes_helpful                 0
votes_funny                   0
weighted_vote_score           0
comment_count                 0
timestamp_created             0
timestamp_updated             0
review_text                 188
review_length                 0
review_id                     0
playtime_ratio                0
review_engagement_score       0
experienced_gamer             0
active_reviewer               0
dtype: int64

Data types:
recommended                    bool
user_id                       int64
user_games_owned              int64
user_num_reviews              int64
playtime_at_review_hours    float64
playtime_total_hours 

,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,votes_helpful,votes_funny,weighted_vote_score,comment_count,timestamp_created,timestamp_updated,review_length,review_id,playtime_ratio,review_engagement_score
count,5.000000e+04,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,5.000000e+04,5.000000e+04,50000.000000,5.000000e+04,50000.000000,50000.000000
mean,7.656120e+16,60.827420,9.950820,571.543931,851.852829,2.905052,1.591480,0.328500,0.503188,0.050300,1.665138e+09,1.667795e+09,90.716380,1.258723e+08,0.671733,1.755730
std,4.050778e+08,287.152493,39.024014,1077.466858,1493.023876,11.375891,49.391639,10.005544,0.024541,0.995324,4.113490e+07,4.204586e+07,251.808146,3.278874e+07,0.302666,53.840778
min,7.656120e+16,0.000000,1.000000,0.083333,0.133333,0.000000,0.000000,0.000000,0.132850,0.000000,1.612215e+09,1.612215e+09,0.000000,8.578226e+07,0.000038,0.000000
25%,7.656120e+16,0.000000,1.000000,31.716667,74.829167,0.000000,0.000000,0.000000,0.500000,0.000000,1.631402e+09,1.633258e+09,8.000000,9.919439e+07,0.448158,0.000000
50%,7.656120e+16,0.000000,3.000000,171.291667,293.391667,0.000000,0.000000,0.000000,0.500000,0.000000,1.650211e+09,1.653607e+09,23.000000,1.140148e+08,0.768498,0.000000
75%,7.656120e+16,51.000000,9.000000,646.425000,957.620833,0.000000,0.000000,0.000000,0.500000,0.000000,1.698674e+09,1.702011e+09,73.000000,1.491942e+08,0.940019,0.000000
max,7.656120e+16,33673.000000,6003.000000,30972.666667,43321.100000,297.600000,7228.000000,1519.000000,0.945723,158.000000,1.761706e+09,1.761737e+09,7832.000000,2.078426e+08,1.062945,7987.500000


---

## 2. Feature Engineering

Prepare features from review text, playtime, and other variables for the binary classifier.

In [ ]:
# Feature engineering imports
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
import re

Feature engineering libraries loaded successfully!


In [ ]:
# ============================================================================
# TEXT FEATURE ENGINEERING
# ============================================================================

logger.info("Starting text feature engineering")

# Handle missing review texts
missing_count = df['review_text'].isnull().sum()
if missing_count > 0:
    logger.warning(f"Found {missing_count} missing review texts, filling with empty string")
    df['review_text'] = df['review_text'].fillna('')

# 1. BASIC TEXT STATISTICS
logger.debug("Extracting basic text statistics")
df['word_count'] = df['review_text'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['review_text'].apply(lambda x: len(str(x)))
df['avg_word_length'] = df['review_text'].apply(
    lambda x: np.mean([len(word) for word in str(x).split()]) if len(str(x).split()) > 0 else 0
)

# 2. EMOTIONAL INDICATORS
logger.debug("Extracting emotional indicators")
df['exclamation_count'] = df['review_text'].apply(lambda x: str(x).count('!'))
df['question_count'] = df['review_text'].apply(lambda x: str(x).count('?'))
df['caps_ratio'] = df['review_text'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()) / (len(str(x)) + 1)
)
df['ellipsis_count'] = df['review_text'].apply(lambda x: str(x).count('...'))

# 3. SENTIMENT ANALYSIS
logger.info("Computing sentiment scores (TextBlob)")

def get_sentiment(text):
    try:
        blob = TextBlob(str(text))
        return blob.sentiment.polarity, blob.sentiment.subjectivity
    except Exception as e:
        logger.debug(f"Sentiment extraction failed: {e}")
        return 0.0, 0.0

sentiment_scores = df['review_text'].apply(get_sentiment)
df['sentiment_polarity'] = sentiment_scores.apply(lambda x: x[0])
df['sentiment_subjectivity'] = sentiment_scores.apply(lambda x: x[1])

# 4. REVIEW QUALITY INDICATORS
logger.debug("Creating review quality indicators")
df['is_detailed_review'] = (df['word_count'] > 50).astype(int)
df['is_substantive'] = ((df['word_count'] > 20) & (df['exclamation_count'] < 5)).astype(int)
df['is_one_word'] = (df['word_count'] <= 3).astype(int)
df['is_all_caps'] = (df['caps_ratio'] > 0.8).astype(int)

logger.info(f"Created 13 text-based features")
logger.debug(f"  - Basic: word_count, char_count, avg_word_length")
logger.debug(f"  - Emotional: exclamation_count, question_count, caps_ratio, ellipsis_count")
logger.debug(f"  - Sentiment: polarity [{df['sentiment_polarity'].min():.2f}, {df['sentiment_polarity'].max():.2f}], subjectivity")
logger.debug(f"  - Quality: is_detailed, is_substantive, is_one_word, is_all_caps")

In [ ]:
# ============================================================================
# NUMERIC & BEHAVIORAL FEATURE ENGINEERING
# ============================================================================

logger.info("Starting numeric and behavioral feature engineering")

# 1. PLAYTIME ENGINEERING
logger.debug("Engineering playtime features")
df['log_playtime'] = np.log1p(df['playtime_total_hours'])
df['log_playtime_at_review'] = np.log1p(df['playtime_at_review_hours'])

# Playtime categories (interpretable bins)
def categorize_playtime(hours):
    """Categorize playtime into interpretable bins for counterfactuals."""
    if hours < 2:
        return 0  # 'refund_window'
    elif hours < 10:
        return 1  # 'early_impression'
    elif hours < 50:
        return 2  # 'casual'
    elif hours < 200:
        return 3  # 'regular'
    elif hours < 500:
        return 4  # 'dedicated'
    elif hours < 1000:
        return 5  # 'hardcore'
    else:
        return 6  # 'addicted'

df['playtime_category'] = df['playtime_total_hours'].apply(categorize_playtime)

# Binary playtime indicators
df['is_refund_window'] = (df['playtime_at_review_hours'] < 2).astype(int)
df['is_veteran'] = (df['playtime_total_hours'] > 100).astype(int)
df['is_hardcore'] = (df['playtime_total_hours'] > 500).astype(int)
df['has_recent_playtime'] = (df['playtime_recent_hours'] > 0).astype(int)

playtime_stats = {
    'refund': (df['playtime_category'] == 0).sum(),
    'veteran': df['is_veteran'].sum(),
    'hardcore': df['is_hardcore'].sum()
}
logger.debug(f"Playtime distribution: {playtime_stats}")

# 2. GAME-SPECIFIC KEYWORD FEATURES
logger.info("Extracting PUBG-specific keyword features")

# Technical issues (negative signals)
df['mentions_cheaters'] = df['review_text'].str.lower().str.contains(
    'cheat|hack|aimbot|wallhack|esp', regex=True, na=False
).astype(int)

df['mentions_bugs'] = df['review_text'].str.lower().str.contains(
    'bug|glitch|broken|crash', regex=True, na=False
).astype(int)

df['mentions_lag'] = df['review_text'].str.lower().str.contains(
    'lag|fps|stutter|freeze|performance|optimiz', regex=True, na=False
).astype(int)

df['mentions_servers'] = df['review_text'].str.lower().str.contains(
    'server|disconnect|ping|connection', regex=True, na=False
).astype(int)

# Positive keywords
df['mentions_fun'] = df['review_text'].str.lower().str.contains(
    'fun|enjoy|love|amazing|awesome|great', regex=True, na=False
).astype(int)

df['mentions_addictive'] = df['review_text'].str.lower().str.contains(
    'addictive|addicted|hooked|cant stop|one more', regex=True, na=False
).astype(int)

# Strong sentiment words
df['has_strong_negative'] = df['review_text'].str.lower().str.contains(
    'hate|terrible|awful|worst|garbage|trash|shit', regex=True, na=False
).astype(int)

df['has_strong_positive'] = df['review_text'].str.lower().str.contains(
    'best|perfect|masterpiece|incredible|outstanding', regex=True, na=False
).astype(int)

# Competitor mentions
df['mentions_competitors'] = df['review_text'].str.lower().str.contains(
    'fortnite|apex|warzone|cod', regex=True, na=False
).astype(int)

keyword_stats = {
    'cheaters': df['mentions_cheaters'].sum(),
    'bugs': df['mentions_bugs'].sum(),
    'lag': df['mentions_lag'].sum(),
    'fun': df['mentions_fun'].sum()
}
logger.debug(f"Keyword mentions: {keyword_stats}")

# 3. INTERACTION FEATURES
logger.debug("Creating interaction features")
df['sentiment_x_playtime'] = df['sentiment_polarity'] * df['log_playtime']
df['wordcount_x_playtime'] = df['word_count'] * df['log_playtime']

# 4. USER BEHAVIOR FEATURES
logger.debug("Engineering user behavior features")
df['log_games_owned'] = np.log1p(df['user_games_owned'])
df['is_collector'] = (df['user_games_owned'] > 100).astype(int)

logger.info(f"Created 25 numeric and behavioral features")
logger.debug(f"  - Playtime: 7 features (log transforms, categories, binary indicators)")
logger.debug(f"  - Keywords: 9 features (PUBG-specific issues and praise)")
logger.debug(f"  - Interactions: 2 features (sentiment × playtime, wordcount × playtime)")
logger.debug(f"  - User behavior: 7 features (games owned, collector status)")

In [ ]:
# ============================================================================
# FINAL FEATURE SELECTION & PREPARATION
# ============================================================================

logger.info("Preparing final feature matrix")

# Define feature groups (for interpretability and counterfactuals)
SENTIMENT_FEATURES = [
    'sentiment_polarity',
    'sentiment_subjectivity',
]

TEXT_FEATURES = [
    'word_count',
    'char_count',
    'avg_word_length',
    'exclamation_count',
    'question_count',
    'caps_ratio',
    'is_detailed_review',
    'is_substantive',
    'is_one_word',
]

PLAYTIME_FEATURES = [
    'log_playtime',
    'log_playtime_at_review',
    'playtime_category',
    'is_refund_window',
    'is_veteran',
    'is_hardcore',
    'has_recent_playtime',
]

KEYWORD_FEATURES = [
    'mentions_cheaters',
    'mentions_bugs',
    'mentions_lag',
    'mentions_servers',
    'mentions_fun',
    'mentions_addictive',
    'has_strong_negative',
    'has_strong_positive',
]

INTERACTION_FEATURES = [
    'sentiment_x_playtime',
    'wordcount_x_playtime',
]

USER_FEATURES = [
    'log_games_owned',
    'is_collector',
    'experienced_gamer',
    'active_reviewer',
]

# Combine all features
ALL_FEATURES = (
    SENTIMENT_FEATURES + 
    TEXT_FEATURES + 
    PLAYTIME_FEATURES + 
    KEYWORD_FEATURES + 
    INTERACTION_FEATURES + 
    USER_FEATURES
)

logger.info(f"Feature summary: {len(ALL_FEATURES)} total features")
logger.debug(f"  Sentiment: {len(SENTIMENT_FEATURES)}")
logger.debug(f"  Text: {len(TEXT_FEATURES)}")
logger.debug(f"  Playtime: {len(PLAYTIME_FEATURES)}")
logger.debug(f"  Keywords: {len(KEYWORD_FEATURES)}")
logger.debug(f"  Interactions: {len(INTERACTION_FEATURES)}")
logger.debug(f"  User: {len(USER_FEATURES)}")

# Create feature matrix X and target y
X = df[ALL_FEATURES].copy()
y = df['recommended'].copy()

# Handle missing values
nan_count = X.isnull().sum().sum()
if nan_count > 0:
    logger.warning(f"Found {nan_count} NaN values in features, filling with 0")
    X = X.fillna(0)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logger.info(f"Train/test split complete")
logger.info(f"  Training: {X_train.shape[0]:,} samples × {X_train.shape[1]} features")
logger.info(f"  Testing:  {X_test.shape[0]:,} samples × {X_test.shape[1]} features")
logger.info(f"  Class distribution (train): {y_train.sum():,} positive ({y_train.mean()*100:.2f}%)")
logger.info(f"  Class distribution (test):  {y_test.sum():,} positive ({y_test.mean()*100:.2f}%)")

# Display feature preview
print("\nFeature matrix preview:")
print(X_train.head())

---

## 3. Model Training

Train the binary classifier to predict whether a user will recommend a game.

In [ ]:
# ============================================================================
# FEATURE ANALYSIS & VALIDATION
# ============================================================================

logger.info("Analyzing feature correlations and importance")

# Calculate correlation with target
feature_correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)

print("\nTop 15 Most Predictive Features (by correlation):")
print("="*70)
for i, (feature, corr) in enumerate(feature_correlations.head(15).items(), 1):
    print(f"{i:2d}. {feature:30s} │ r = {corr:.4f}")
print("="*70)

# Log top features
top_features = feature_correlations.head(5).to_dict()
logger.info(f"Top 5 features: {list(top_features.keys())}")

# Check for multicollinearity
logger.debug("Checking for multicollinearity")
corr_matrix = X_train.corr().abs()
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.8:
            high_corr_pairs.append(
                (corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j])
            )

if high_corr_pairs:
    logger.warning(f"Found {len(high_corr_pairs)} highly correlated pairs (|r| > 0.8)")
    for f1, f2, corr in high_corr_pairs[:3]:
        logger.debug(f"  {f1} ↔ {f2}: {corr:.3f}")
else:
    logger.info("No major multicollinearity detected (all |r| < 0.8)")

logger.info("Feature engineering complete - ready for model training")

In [ ]:
def train_classifier(X_train, y_train, **kwargs):
    """
    Train binary classifier for game recommendation prediction.
    
    Parameters
    ----------
    X_train : array-like
        Training features
    y_train : array-like
        Training labels
    **kwargs : dict
        Additional model parameters
        
    Returns
    -------
    model
        Trained classifier
    """
    # TODO: Implement training logic
    # - Choose classifier (LogisticRegression, RandomForest, XGBoost, etc.)
    # - Handle class imbalance if needed
    # - Fit model
    # - Return trained model
    
    pass

In [ ]:
# TODO: Train the model
# model = train_classifier(X_train, y_train)

pass

---

## 4. Prediction Function (with Counterfactuals)

Create a function that:
- Takes a user + game input
- Outputs binary prediction (yes/no they'll like it)
- If prediction is negative: generates counterfactual explanations

In [ ]:
def predict_with_counterfactuals(model, user_features, threshold=0.5):
    """
    Predict game recommendation and generate counterfactuals if negative.
    
    Parameters
    ----------
    model : classifier
        Trained binary classifier
    user_features : dict or array-like
        User and game features
    threshold : float
        Classification threshold
        
    Returns
    -------
    dict
        Dictionary containing:
        - 'prediction': bool (True if recommended)
        - 'probability': float (model confidence)
        - 'counterfactuals': list (if prediction is negative)
    """
    # TODO: Implement prediction logic
    # - Get prediction probability
    # - Make binary prediction
    # - If negative: generate counterfactual explanations
    #   ("If you played X more hours, prediction would be positive")
    #   ("If review text contained Y sentiment, prediction would be positive")
    
    pass

In [ ]:
def generate_counterfactuals(model, original_features, feature_names):
    """
    Generate counterfactual explanations for negative predictions.
    
    Parameters
    ----------
    model : classifier
        Trained model
    original_features : array-like
        Original feature values
    feature_names : list
        Names of features
        
    Returns
    -------
    list
        List of counterfactual explanations
    """
    # TODO: Implement counterfactual generation
    # - Try modifying each feature
    # - Find minimal changes that flip prediction
    # - Return human-readable explanations
    
    pass

In [ ]:
# TODO: Demo prediction function
# - Create sample user input
# - Get prediction + counterfactuals
# - Display results

pass

---

## 5. Model Evaluation

Assess model performance using appropriate metrics.

In [ ]:
# TODO: Evaluate model on test set
# - Accuracy
# - Precision, Recall, F1-Score
# - ROC-AUC
# - Confusion Matrix

pass

In [ ]:
# TODO: Visualize results
# - Confusion matrix heatmap
# - ROC curve
# - Feature importance plot

pass

In [ ]:
# TODO: Print final evaluation summary

pass

---

## Summary

This notebook provides a complete pipeline for interpretable game recommendation predictions:

1. **Data Loading & Exploration** - Loaded and analyzed Steam review data
2. **Feature Engineering** - Extracted features from text and numeric data
3. **Model Training** - Trained binary classifier
4. **Prediction Function** - Generated predictions with counterfactual explanations
5. **Model Evaluation** - Assessed performance metrics

---

**Next Steps:**
- Implement remaining sections
- Tune model hyperparameters
- Improve counterfactual generation logic
- Add visualizations